# DataForge Arena — GRPO Training on Colab

**Meta × PyTorch × Hugging Face × Scaler OpenEnv Hackathon 2026**

Use **Runtime → Change runtime type → T4 GPU** before running.


## 1 — Clone Repository


In [ ]:
import os, subprocess

REPO = '/content/dataforge-arena'
if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', 'origin/master'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/vivekyarra/dataforge-arena.git', REPO], check=True)

os.chdir(REPO)
print('Working dir:', os.getcwd())
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())


## 2 — Install Dependencies


In [ ]:
%%capture install_log
import subprocess, sys, os

# Use Colab's pre-installed CUDA 12.4 torch stack — do NOT upgrade torch.
# Only install the RL/LLM packages on top.
packages = [
    'unsloth[colab-new]',
    'trl==0.24.0',
    'peft>=0.14.0',
    'transformers',
    'datasets',
    'accelerate',
    'bitsandbytes',
    'xformers',
    'mergekit>=0.1.4',
    'openenv',
    'faker',
    'gradio>=4.26.0',
    'scipy',
    'matplotlib',
    'huggingface_hub',
    'httpx',
    'pytest',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'unsloth_zoo'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)
print('Install complete.')


In [ ]:
import torch, transformers, trl, unsloth
print(f'torch={torch.__version__}  cuda={torch.version.cuda}  gpu={torch.cuda.get_device_name(0)}')
print(f'transformers={transformers.__version__}  trl={trl.__version__}  unsloth={unsloth.__version__}')
print('All imports OK.')


## 3 — Generate Clean Data


In [ ]:
!python training/generate_data.py


## 4 — GPU & Model Config


In [ ]:
import subprocess, sys

!nvidia-smi -L

check = """
import json
from training.model_config import detect_gpu, select_model, select_precision
gpu = detect_gpu()
cfg = select_model(gpu)
prec = select_precision(gpu)
print(json.dumps({'gpu': gpu, 'model': cfg, 'precision': prec}, indent=2))
"""
subprocess.run([sys.executable, '-c', check], check=True)


## 5 — Run Tests


In [ ]:
!python -m pytest tests/ -q --tb=short


## 6 — Observation Sanity Check


In [ ]:
import subprocess, sys

sanity = """
import json, pandas as pd
from environment.corruptor import Corruptor
from environment.env import DataForgeEnv
from environment.schemas import HEALTHCARE_SCHEMA
from training.prompt import build_prompt

clean_df = pd.read_csv('data/healthcare_clean.csv')
env = DataForgeEnv(Corruptor(), HEALTHCARE_SCHEMA, clean_df)
obs = env.reset()
prompt = build_prompt(obs)
print(f'Schema: {obs.schema_str[:120]}...')
print(f'Prompt tokens (est): {len(prompt)//4}')
print(f'Difficulty: {obs.difficulty}  Errors: {obs.total_errors}')
print('Sanity check passed.')
"""
subprocess.run([sys.executable, '-c', sanity], check=True)


## 7 — GRPO Training

This runs `training/train_grpo.py` in a subprocess so Unsloth patches load cleanly.
On T4: ~40-55 minutes. On A100: ~15-20 minutes.


In [ ]:
import subprocess, sys, time

start = time.time()
result = subprocess.run(
    [sys.executable, 'training/train_grpo.py'],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
elapsed = time.time() - start

# Always print output so we can debug
print(result.stdout[-5000:] if len(result.stdout) > 5000 else result.stdout)
print(f'\n--- Finished in {elapsed/60:.1f} minutes (exit code {result.returncode}) ---')

if result.returncode != 0:
    print('\n⚠️  Training failed. Full output above. Common fixes:')
    print('  1. Runtime → Restart session, then rerun from Cell 1')
    print('  2. Ensure GPU runtime is selected (T4 minimum)')
    print('  3. Check for CUDA OOM — reduce batch_size in model_config.py')


## 8 — Evaluation


In [ ]:
!python eval/evaluate.py --agent-mode grpo --model-path outputs/dataforge-surgeon --episodes 20 --tier 1 --steps 5 --seed 7

from pathlib import Path
print(Path('eval/results.json').read_text(encoding='utf-8'))


## 9 — Training Curves


In [ ]:
import subprocess, sys
from pathlib import Path

out = Path('logs/training_curve.png')
subprocess.run([sys.executable, 'scripts/plot_training.py', '--log', 'logs/training_log.csv', '--out', str(out)], check=True)

try:
    from IPython.display import Image, display
    display(Image(filename=str(out)))
except Exception as e:
    print(f'Display skipped: {e}')


## 10 — Save & Download


In [ ]:
import shutil
from pathlib import Path

export = Path('colab_export')
if export.exists(): shutil.rmtree(export)
export.mkdir(parents=True)

for p in ['outputs/dataforge-surgeon', 'eval/results.json', 'logs/training_log.csv', 'logs/training_curve.png']:
    src = Path(p)
    if not src.exists(): continue
    dst = export / src.name
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)

zip_path = shutil.make_archive('dataforge-arena-export', 'zip', export)
print(f'Export: {zip_path}')

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print('Auto-download skipped. Download manually from the file browser.')


## 11 — (Optional) Hugging Face Upload


In [ ]:
import os
from pathlib import Path
from huggingface_hub import HfApi, create_repo, login, notebook_login

HF_REPO_ID = ''        # e.g. 'Vivek567/dataforge-arena-model'
UPLOAD_TO_HF = False   # Set True when ready
HF_TOKEN = os.getenv('HF_TOKEN', '')

if not UPLOAD_TO_HF:
    print('Skipping HF upload. Set UPLOAD_TO_HF=True and HF_REPO_ID to enable.')
else:
    if not HF_REPO_ID: raise ValueError('Set HF_REPO_ID first.')
    login(token=HF_TOKEN, add_to_git_credential=True) if HF_TOKEN else notebook_login()
    model_dir = Path('outputs/dataforge-surgeon')
    if not model_dir.exists(): raise FileNotFoundError('No checkpoint at outputs/dataforge-surgeon')
    create_repo(HF_REPO_ID, repo_type='model', exist_ok=True)
    api = HfApi()
    api.upload_folder(folder_path=str(model_dir), repo_id=HF_REPO_ID, repo_type='model', path_in_repo='dataforge-surgeon')
    for f in ['eval/results.json', 'logs/training_log.csv', 'logs/training_curve.png']:
        if Path(f).exists():
            api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=HF_REPO_ID, repo_type='model')
    print(f'Upload complete: https://huggingface.co/{HF_REPO_ID}')
